In [ ]:
!pip install torch torchvision torchaudio scikit-learn tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import numpy as np
import torch

ROOT           = "/content/drive/MyDrive/Hackenza_MUPlovers"
FEATURE_DIR    = os.path.join(ROOT, "cache/features_normalized")
TRAIN_MANIFEST = os.path.join(ROOT, "data/train_manifest_split.csv")
VAL_MANIFEST   = os.path.join(ROOT, "data/val_manifest_split.csv")

print("Feature dir exists:", os.path.exists(FEATURE_DIR))
print("Train manifest exists:", os.path.exists(TRAIN_MANIFEST))
print("Val manifest exists:", os.path.exists(VAL_MANIFEST))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
from torch.utils.data import Dataset

class NativeDataset(Dataset):
    def __init__(self, manifest_csv, feature_dir):
        self.df          = pd.read_csv(manifest_csv)
        self.feature_dir = feature_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        file_id = str(row["file_id"])
        label   = float(row["label"])

        feature_path = os.path.join(self.feature_dir, f"{file_id}.npy")
        x = np.load(feature_path)
        x = torch.tensor(x, dtype=torch.float32)

        return x, torch.tensor(label, dtype=torch.float32)

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = [seq.shape[0] for seq in sequences]
    padded  = pad_sequence(sequences, batch_first=True)

    mask = torch.zeros(padded.shape[:2])
    for i, length in enumerate(lengths):
        mask[i, :length] = 1

    labels = torch.stack(labels)
    return padded, mask, labels

In [ ]:
from torch.utils.data import DataLoader

train_dataset = NativeDataset(TRAIN_MANIFEST, FEATURE_DIR)
val_dataset   = NativeDataset(VAL_MANIFEST,   FEATURE_DIR)

train_loader = DataLoader(
    train_dataset,
    batch_size = 8,
    shuffle    = True,
    collate_fn = collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size = 8,
    shuffle    = False,
    collate_fn = collate_fn
)

print("Train size:", len(train_dataset))
print("Val size  :", len(val_dataset))

In [ ]:
# Compute pos_weight automatically
train_df       = pd.read_csv(TRAIN_MANIFEST)
num_native     = (train_df['label'] == 1).sum()
num_non_native = (train_df['label'] == 0).sum()

pos_weight = torch.tensor([num_non_native / num_native], dtype=torch.float32).to(device)

print(f"Native samples    : {num_native}")
print(f"Non-native samples: {num_non_native}")
print(f"pos_weight        : {pos_weight.item():.4f}")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class GRUOnlyModel(nn.Module):
    """
    GRU only — no attention.
    Takes last hidden state of GRU as the summary vector.
    """
    def __init__(self, input_dim=783, hidden_dim=64):
        super().__init__()
        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            batch_first   = True,
            bidirectional = True
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x, mask):
        # Run GRU
        out, _ = self.gru(x)   # out: [B, T, hidden*2]

        # Mean pooling over real chunks only (using mask)
        mask_expanded = mask.unsqueeze(-1).expand_as(out)
        sum_out       = (out * mask_expanded).sum(dim=1)
        count         = mask.sum(dim=1, keepdim=True).clamp(min=1)
        pooled        = sum_out / count   # [B, hidden*2]

        logits = self.classifier(pooled).squeeze(-1)
        return logits

In [ ]:
import torch.optim as optim

model     = GRUOnlyModel(783).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience = 3,
    factor   = 0.5
)

print("Model ready!")
print("Total parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

def train_epoch():
    model.train()
    total_loss = 0

    for x, mask, y in tqdm(train_loader):
        x, mask, y = x.to(device), mask.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x, mask)
        loss   = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(train_loader)


def evaluate():
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for x, mask, y in val_loader:
            x, mask, y = x.to(device), mask.to(device), y.to(device)

            logits = model(x, mask)
            loss   = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, zero_division=0)
    val_loss = total_loss / len(val_loader)

    return acc, f1, val_loss

In [ ]:
EPOCHS   = 50
best_acc = 0
best_f1  = 0

for epoch in range(EPOCHS):
    loss              = train_epoch()
    acc, f1, val_loss = evaluate()

    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss : {loss:.4f}")
    print(f"  Val Loss   : {val_loss:.4f}")
    print(f"  Val Acc    : {acc:.4f}")
    print(f"  Val F1     : {f1:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_f1  = f1
        os.makedirs(os.path.join(ROOT, 'runs'), exist_ok=True)
        torch.save(model.state_dict(), os.path.join(ROOT, 'runs/best_model_gru_only.pt'))
        print(f"  Best model saved!")

    print("-" * 40)

print(f"\nBest Accuracy: {best_acc:.4f}")
print(f"Best F1      : {best_f1:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x, mask, y in val_loader:
        x, mask = x.to(device), mask.to(device)
        logits = model(x, mask)
        probs  = torch.sigmoid(logits)
        preds  = (probs > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())

print(classification_report(all_labels, all_preds, target_names=['Non-Native', 'Native']))
print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))